In [14]:
import re
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from langchain_huggingface import HuggingFaceEmbeddings , ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.prompts import PromptTemplate

load_dotenv()

True

In [2]:

_PLAIN_ID_RE = re.compile(r"^[A-Za-z0-9_-]{11}$")

def extract_video_id(url_or_id: str) -> str:
    """
    Parse any YouTube URL or plain 11-char ID and return the video ID.

    Supports:
      https://www.youtube.com/watch?v=dQw4w9WgXcQ
      https://youtu.be/dQw4w9WgXcQ
      https://youtube.com/shorts/dQw4w9WgXcQ
      dQw4w9WgXcQ  (plain ID)
    """
    value = url_or_id.strip()

    if _PLAIN_ID_RE.match(value):
        return value

    parsed = urlparse(value)
    host = parsed.netloc.lower().removeprefix("www.")

    if host in ("youtube.com", "m.youtube.com"):
        qs = parse_qs(parsed.query)
        if "v" in qs and qs["v"]:
            vid = qs["v"][0]
            if _PLAIN_ID_RE.match(vid):
                return vid
        parts = [p for p in parsed.path.split("/") if p]
        if len(parts) >= 2 and parts[0] in ("shorts", "embed", "v", "e"):
            vid = parts[1]
            if _PLAIN_ID_RE.match(vid):
                return vid

    elif host == "youtu.be":
        vid = parsed.path.lstrip("/").split("?")[0]
        if _PLAIN_ID_RE.match(vid):
            return vid

    raise ValueError(f"Could not extract a YouTube video ID from: '{value}'")


def format_timestamp(seconds: float) -> str:
    """Convert raw seconds → MM:SS string."""
    total = int(seconds)
    return f"{total // 60:02d}:{total % 60:02d}"


def youtube_link(video_id: str, seconds: float) -> str:
    """Return a YouTube deep-link that starts at the given second."""
    return f"https://youtu.be/{video_id}?t={int(seconds)}"



print( "dQw4w9WgXcQ: ",extract_video_id("dQw4w9WgXcQ"))
print("https://youtu.be/dQw4w9WgXcQ: ",extract_video_id("https://youtu.be/dQw4w9WgXcQ"))
print("https://www.youtube.com/watch?v=dQw4w9WgXcQ: ",extract_video_id("https://www.youtube.com/watch?v=dQw4w9WgXcQ"))
print("formated time: ",format_timestamp(180))
print("youtube_link with timestamp: ",youtube_link("a3C1DMswClQ",12))

dQw4w9WgXcQ:  dQw4w9WgXcQ
https://youtu.be/dQw4w9WgXcQ:  dQw4w9WgXcQ
https://www.youtube.com/watch?v=dQw4w9WgXcQ:  dQw4w9WgXcQ
formated time:  03:00
youtube_link with timestamp:  https://youtu.be/a3C1DMswClQ?t=12


In [ ]:

PREFERRED_LANGS = ["en", "en-US", "en-GB", "hi"]

def fetch_transcript(video_id: str) -> list:
    """
    Fetch the best available transcript for a video.

    Returns a list of transcript entries where each entry has:
      .text      - spoken text
      .start     - offset in seconds from start of video
      .duration  - duration of the segment in seconds
    """
    api = YouTubeTranscriptApi()

    try:
        transcript_list = api.list(video_id)
    except TranscriptsDisabled:
        raise RuntimeError(f"Transcripts are disabled for video '{video_id}'.")
    except Exception as e:
        raise RuntimeError(f"Could not fetch transcript for '{video_id}': {e}")

    print("Available transcripts:")
    for t in transcript_list:
        print(f"  {t.language_code} - {t.language}")

    try:
        transcript = transcript_list.find_transcript(PREFERRED_LANGS).fetch()
        print("Preferred language transcript fetched.")
    except NoTranscriptFound:
        transcript = next(iter(transcript_list)).fetch()
        print("Preferred language not found — using fallback.")

    return transcript


def full_text_from_transcript(transcript: list) -> str:
    """Join all transcript segments into one string (used for summarisation)."""
    return " ".join(entry.text for entry in transcript)


In [4]:
data = fetch_transcript("a3C1DMswClQ")
data

Available transcripts:
  en – English (auto-generated)
Preferred language transcript fetched.


FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='backend is huge and if we start', start=0.08, duration=3.92), FetchedTranscriptSnippet(text='discussing every single component that', start=2.159, duration=3.68), FetchedTranscriptSnippet(text='could be part of it we will be stuck', start=4.0, duration=4.36), FetchedTranscriptSnippet(text='here for years so what we will do is we', start=5.839, duration=5.081), FetchedTranscriptSnippet(text='will only discuss those topics which are', start=8.36, duration=4.64), FetchedTranscriptSnippet(text="used in majority of the code bases let's", start=10.92, duration=4.48), FetchedTranscriptSnippet(text="say 90% of them with that in mind let's", start=13.0, duration=5.32), FetchedTranscriptSnippet(text='talk about HTTP protocol the medium', start=15.4, duration=4.84), FetchedTranscriptSnippet(text='through which our browsers talk to our', start=18.32, duration=4.199), FetchedTranscriptSnippet(text='servers either to send data or to', start=

In [10]:
def build_timed_chunks(
    transcript: list,
    chunk_size: int = 1000,
    overlap: int = 200,
) -> list[Document]:
    """
    Split the transcript into overlapping text chunks while preserving
    the start timestamp of each chunk as Document metadata.

    Args:
        transcript  : list of transcript entries (.text, .start)
        chunk_size  : max characters per chunk
        overlap     : how many characters from the end of the previous
                      chunk to repeat at the start of the next one

    Returns:
        List of LangChain Documents with metadata["start"] set.
    """
    documents: list[Document] = []
    current_text = ""
    current_start: float = 0.0

    for snippet in transcript:
        word = snippet.text.strip()
        start: float = snippet.start

        if not current_text:
            current_start = start

        if len(current_text) + len(word) + 1 > chunk_size:
            documents.append(Document(
                page_content=current_text.strip(),
                metadata={"start": current_start},
            ))
            
            overlap_text = current_text[-overlap:].strip() if overlap else ""
            current_text = overlap_text + " " + word
            current_start = start
        else:
            current_text += " " + word


    if current_text.strip():
        documents.append(Document(
            page_content=current_text.strip(),
            metadata={"start": current_start},
        ))

    print(f"Created {len(documents)} timed chunks.")
    return documents

In [11]:
chunks=build_timed_chunks(data)

Created 87 timed chunks.


In [13]:
chunks[0]

Document(metadata={'start': 0.08}, page_content="backend is huge and if we start discussing every single component that could be part of it we will be stuck here for years so what we will do is we will only discuss those topics which are used in majority of the code bases let's say 90% of them with that in mind let's talk about HTTP protocol the medium through which our browsers talk to our servers either to send data or to receive data from it and as I said there are a lot of other ways and protocols which clients and servers use to communicate with each other and HTTP being one of the most used ones we will focus on that now there are two ideas which are at the heart of HTTP protocol the first one being tessness what does it mean tessness basically means it has no memory of past interactions so each HTTP request carries all the necessary information for the server to process it such as headers or URLs and methods which we will see in a bit and after the server responds it forgets abo

In [15]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def build_vector_store(chunks: list[Document]) -> FAISS:
    """Embed chunks and return a FAISS vector store."""

    print(f"Building vector store for {len(chunks)} chunks …")
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)  
    store = FAISS.from_documents(chunks, embeddings)
    print("Vector store ready.")
    return store


def get_retriever(store: FAISS, k: int = 4):
    """Return a similarity retriever that fetches the top-k chunks."""
    
    return store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )


In [16]:
LLM_REPO_ID = "deepseek-ai/DeepSeek-V3.1-Terminus"

def get_chat_model() -> ChatHuggingFace:
    """Initialise and return the chat model."""

    print(f"Loading LLM: {LLM_REPO_ID} …")
    llm = HuggingFaceEndpoint(repo_id=LLM_REPO_ID, 
                              task="conversational")
    model = ChatHuggingFace(llm=llm)
    print("LLM ready.")
    return model


In [17]:
model=get_chat_model()

Loading LLM: deepseek-ai/DeepSeek-V3.1-Terminus …


c:\Users\Govin\Downloads\yt_chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM ready.


In [18]:
model.invoke("hi")

AIMessage(content='Hello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': ChatCompletionOutputUsage(completion_tokens=12, prompt_tokens=11, total_tokens=23, prompt_tokens_details=None, completion_tokens_details=None), 'model': '', 'finish_reason': 'stop'}, id='run--5ecd860a-8176-4f52-83bc-8fcf13453b94-0')